# 05 — Performance Tuning & Optimization

Practical tuning knobs for Fabric Spark jobs, in the order you should usually try them:
partitioning & file layout, caching, join strategy, AQE, skew handling, and pool/session
configuration. Use the **Spark Application Detail / Monitoring hub** in Fabric alongside this
notebook to see the before/after effect of each change.


## 1. Partitioning: `repartition` vs `coalesce`

In [ ]:
df = spark.read.table("silver_patient_visits")
print("Input partitions:", df.rdd.getNumPartitions())

# repartition() triggers a full shuffle — use to INCREASE parallelism or fix skew
df_wide = df.repartition(64, "department")

# coalesce() avoids a shuffle — use to REDUCE partitions before a write (fewer small files)
df_narrow = df.coalesce(8)

print("After repartition:", df_wide.rdd.getNumPartitions())
print("After coalesce:", df_narrow.rdd.getNumPartitions())


## 2. Caching / persisting reused DataFrames

In [ ]:
from pyspark import StorageLevel

df_base = spark.read.table("silver_patient_visits").filter("age IS NOT NULL")
df_base.persist(StorageLevel.MEMORY_AND_DISK)
df_base.count()   # materializes the cache (actions trigger caching, not the .persist() call itself)

# ... reuse df_base in several downstream aggregations ...
dept_counts = df_base.groupBy("department").count()
age_stats = df_base.groupBy("department").avg("age")

# Always release cache you no longer need — it competes for executor memory
df_base.unpersist()


## 3. Join strategy: broadcast vs shuffle joins

In [ ]:
from pyspark.sql import functions as F

# Check the physical plan to see which join Spark actually picked
df_departments = spark.read.table("gold_department_dim")
df_joined = df_base.join(df_departments, "department")
df_joined.explain(mode="formatted")

# Force a broadcast for a dimension table you know is small (< ~100MB is typical for Fabric pools)
df_joined_hint = df_base.join(F.broadcast(df_departments), "department")

# Raise/lower the auto-broadcast threshold globally for the session
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 100 * 1024 * 1024)  # 100 MB


## 4. Adaptive Query Execution (AQE)

AQE is on by default in Fabric runtimes, but it's worth knowing the levers — especially when
diagnosing skewed joins in the Spark UI's SQL tab.

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")   # merges small shuffle partitions
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")             # splits skewed partitions automatically
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "128m")


## 5. Handling skew manually with salting

When AQE's automatic skew handling isn't enough (e.g. a handful of keys dominate a join),
salting spreads the hot keys across multiple synthetic partitions.

In [ ]:
import pyspark.sql.functions as F

SALT_BUCKETS = 10

df_fact_salted = df_base.withColumn("salt", (F.rand() * SALT_BUCKETS).cast("int"))

df_dim_exploded = (
    df_departments
    .withColumn("salt", F.explode(F.array([F.lit(i) for i in range(SALT_BUCKETS)])))
)

df_salted_join = df_fact_salted.join(
    df_dim_exploded, on=["department", "salt"]
).drop("salt")


## 6. Write-time optimization: file sizing and V-Order

In [ ]:
# Target ~128-256MB Delta files instead of thousands of tiny ones
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")   # auto-compacts during write
spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")
spark.conf.set("spark.sql.parquet.vorder.enabled", "true")               # Fabric-specific read acceleration

df_joined.write.format("delta").mode("overwrite").saveAsTable("gold_patient_dept_joined")


## 7. Choosing the right Spark pool / session configuration

In [ ]:
%%configure -f
{
    "conf": {
        "spark.sql.shuffle.partitions": "200",
        "spark.dynamicAllocation.enabled": "true",
        "spark.dynamicAllocation.minExecutors": "2",
        "spark.dynamicAllocation.maxExecutors": "10",
        "spark.sql.adaptive.enabled": "true"
    }
}


## 8. Quick diagnostic checklist
- Too many small files? → `OPTIMIZE`, enable auto-compaction, lower shuffle partitions for small data
- One task takes forever? → check the Spark UI Stages tab for skew, consider salting
- Job spends most time in GC? → cache less, increase executor memory, avoid wide Python UDFs
- Slow join? → check `explain()` for `BroadcastHashJoin` vs `SortMergeJoin`

Next notebook: **06 — Structured Streaming in Fabric**.